# M2. 헤드와 손실 함수 비교 — 본체는 고정, 위층만 바꾸기

**질문 1 (CRF 헤드)**: EDA에서 개체의 74%가 조사·괄호 앞에서 끝나 **경계 오류**가 많을 것으로 예상했다. 꼬리표 순서 규칙을 배우는 CRF가 경계를 더 정확히 잡는가?

**질문 2 (태그 가중 손실)**: NW 0.1%, TR 0.8%, FD 1.8%처럼 드문 태그가 있다. 드문 태그를 틀릴 때 손실을 키우면 macro F1이 오르는가?

본체 모델은 `BASE` 하나로 고정한다(기본 klue/roberta-base. M1에서 더 나은 모델이 나오면 바꾼다). 비교 기준은 M1의 같은 본체 결과다.

## 0. 공통 준비 (모든 노트북 같음)
Colab → 런타임 → 런타임 유형 변경 → GPU(T4 이상). 구글 드라이브를 연결하면 이 PC의 프로젝트 폴더가 `/content/drive/Othercomputers/내 PC/...` 아래에 보인다. 경로가 다르면 왼쪽 파일 탭에서 찾아 `PROJECT` 만 고친다.

공용 코드 `src/ner_lib.py`(학습)와 `src/evaluate.py`(팀 공용 채점기)를 불러온다. 정렬·복원·채점 규칙이 모든 노트북에서 같아야 비교가 공정하기 때문이다.

**저장되는 것**: 실험마다 `reports/results/{실험}.json`(에폭별 dev 점수 포함), `preds/`(예측), `scores/`(채점), `errors/`(오답), 그리고 `runs.jsonl`(실행 기록 한 줄). 그림은 R0 노트북이 이 파일들로 그린다.
**끊겼을 때**: 같은 셀을 다시 실행하면 끝난 실험은 건너뛰고, 진행 중이던 실험은 마지막 에폭 체크포인트부터 이어서 학습한다.

**최종 가중치(10/23부터 항상 저장)**: 학습이 끝나면 예측·채점보다 먼저 `내 드라이브/ner_models/{실험}/` 에 가중치(model.safetensors 또는 crf_model_state.pt)와 토크나이저를 저장하고, 다시 읽어 학습된 값과 같은지 확인한다. 확인에 실패하면 체크포인트를 지우지 않고 멈춘다. 런타임 반납 전에도 한 번 더 확인한다.

In [1]:
!pip -q install -U transformers datasets accelerate pytorch-crf kiwipiepy
import sys, os, json, torch
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/Othercomputers/내 PC/NLP/프로젝트1/공통2_개체명인식_NER"
sys.path.insert(0, f"{PROJECT}/src")
import importlib, ner_lib; importlib.reload(ner_lib)
MODELS = "/content/drive/MyDrive/ner_models"   # 최종 가중치(실험마다 폴더). 학습 직후 저장하고 다시 읽어 확인한다. 지우지 말 것
ner_lib.DEFAULTS.update(model_dir=MODELS)
from ner_lib import run_experiment, load_results, results_table, upper_bound, load_jsonl, prepare_aihub_model
SPLIT   = f"{PROJECT}/data/split_v1"          # dev 1만 / test 1만 (모든 버전 공통)
TRAIN   = f"{PROJECT}/data/split_v3/train.jsonl"   # 팀 기준 학습 데이터: 30만 (2026-10-22 결정)
SUFFIX  = "_300k"                              # 30만으로 학습한 실험 이름 꼬리표
RESULTS = f"{PROJECT}/reports/results"        # 결과 json이 모이는 곳 (팀원 모두 같은 폴더)
CKPT    = "/content/drive/MyDrive/ner_ckpt"    # 에폭 체크포인트. 끊겨도 이어서 학습. '내 드라이브'라 PC로 동기화되지 않음
MEMBER  = "sejin_choi"                          # ← 실행한 사람 이름, 영어로 (결과 기록 runs.jsonl에 남음)
ner_lib.DEFAULTS.update(ckpt_dir=CKPT, member=MEMBER)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 27.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 50.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB


## 1. CRF 헤드는 무엇을 더 하나

**소프트맥스 헤드(기본)**: 토큰마다 15개 꼬리표 점수를 내고 **각자 따로** 가장 높은 것을 고른다. 그래서 `O` 다음에 갑자기 `I-OG` 가 오는 것처럼 BIO 규칙에 안 맞는 순서도 나올 수 있다.

**CRF(Conditional Random Field)**: 문장 전체 꼬리표 순서 $y_1..y_n$ 의 점수를
$$\text{score}(y) = \sum_i E_{i, y_i} + \sum_i T_{y_{i-1}, y_i}$$
로 계산한다. $E$ 는 본체가 낸 토큰별 점수(emission), $T$ 는 **꼬리표에서 꼬리표로 넘어가는 점수(transition, 15×15 표)** 다. 학습하면 $T_{O \to I\text{-}OG}$ 처럼 말이 안 되는 이동은 아주 낮은 점수가 된다.
- 학습: 정답 순서의 확률 $p(y|x) = e^{\text{score}(y)} / \sum_{y'} e^{\text{score}(y')}$ 를 높인다. 분모의 모든 순서 합은 동적계획법(forward 알고리즘)으로 $O(n \cdot 15^2)$ 에 계산한다.
- 예측: 점수가 가장 높은 순서 하나를 **비터비(Viterbi)** 알고리즘으로 찾는다.

구현: `src/ner_lib.py` 의 `build_crf_model`. [CLS]와 [SEP]은 CRF 계산에서 뺀다.

## 2. 태그 가중 손실은 무엇을 바꾸나
기본 교차 엔트로피는 모든 토큰을 같은 무게로 센다. 토큰의 대부분이 `O` 라 모델이 "웬만하면 O"로 기우는 경향이 생긴다. 가중 손실은 꼬리표마다 무게 $w_c = (\text{전체 토큰}/\text{그 꼬리표 토큰})^{1/2}$ 를 곱한다(최대 10배로 제한, O는 1). 드문 태그를 틀리면 손실이 커져 더 신경 쓰게 된다. 대신 정밀도가 떨어질 수 있어서 P와 R을 함께 본다.

In [2]:
ner_lib.DEFAULTS["stage"] = "M2_헤드손실"
BASE = "klue/roberta-base"          # M1 결과로 더 나은 본체가 정해지면 바꾼다
COMMON = dict(train=TRAIN, dev=f"{SPLIT}/dev.jsonl", test=f"{SPLIT}/test.jsonl",
              out_dir=RESULTS, model=BASE, max_len=512, lr=5e-5, epochs=3, batch=16, seed=42)
tag = BASE.split("/")[-1]
VARIANTS = {
    f"M2_{tag}_crf{SUFFIX}":      dict(head="crf"),
    f"M2_{tag}_weighted{SUFFIX}": dict(loss="weighted"),
}

In [3]:
for exp, v in VARIANTS.items():
    run_experiment({**COMMON, **v, "exp": exp})

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

...

  [epoch 2] dev F1 0.7224 macro 0.5655


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [epoch 3] dev F1 0.7338 macro 0.5775


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[M2_roberta-base_weighted_300k] dev F1 0.7338 | test F1 0.7202 | 학습 2902.0s (310.2 문장/s)


## 3. 결과 비교
기준(소프트맥스, 일반 손실)은 M1의 같은 본체 결과다. 보는 법:
- CRF: 전체 F1과 함께 **경계 오류가 줄었는지**를 본다(아래 셀). 오류 중 "태그는 맞는데 범위만 다른" 비율이 줄면 CRF가 제 역할을 한 것이다.
- 가중 손실: `macro_F1` 과 `F1_NW`, `F1_TR`, `F1_FD` 가 올랐는지, 대신 `test_P` 가 얼마나 떨어졌는지.

In [4]:
keep = [r for r in load_results(RESULTS) if r["exp"].startswith("M2_") or r["exp"] == "M1_klue_roberta"]
results_table(keep)[["exp", "test_F1", "test_P", "test_R", "macro_F1", "F1_NW", "F1_TR", "F1_FD", "train_min"]]

,exp,test_F1,test_P,test_R,macro_F1,F1_NW,F1_TR,F1_FD,train_min
0,M1_klue_roberta,0.6866,0.6789,0.6945,0.4681,0.0800,0.3745,0.4188,29.8
1,M2_roberta-base_crf,0.6984,0.6961,0.7008,0.4828,0.0833,0.4262,0.4173,60.6
2,M2_roberta-base_crf_200k,0.7383,0.7392,0.7374,0.5867,0.5000,0.5079,0.4391,119.1
3,M2_roberta-base_crf_300k,0.7474,0.7473,0.7476,0.6148,0.6471,0.4836,0.4505,174.4
4,M2_roberta-base_weighted,0.6780,0.6840,0.6721,0.5094,0.3889,0.3986,0.3903,29.8
5,M2_roberta-base_weighted_300k,0.7202,0.7278,0.7127,0.5957,0.7000,0.4375,0.4246,48.4


In [5]:
# 오답 유형 수 (공용 채점기가 dev 기준으로 저장). 경계 오류가 줄었으면 CRF가 제 역할을 한 것
import pandas as pd
pd.DataFrame({r["exp"]: json.load(open(f"{RESULTS}/scores/{r['exp']}_dev.json", encoding="utf-8"))["error_counts"]
              for r in keep}).T

,경계,유형,경계+유형,오탐,미탐
M1_klue_roberta,1194,242,368,2351,2068
M2_roberta-base_crf,1087,262,299,2233,2135
M2_roberta-base_crf_200k,861,261,225,2021,1862
M2_roberta-base_crf_300k,891,218,230,1874,1763
M2_roberta-base_weighted,1224,247,375,2034,2336
M2_roberta-base_weighted_300k,1118,220,285,1727,2029


## 가중치 저장 확인
이번에 돌린 실험마다 `내 드라이브/ner_models/{실험}/` 에 가중치가 있는지 확인한다. 없으면 에러로 멈춘다.

In [ ]:
ner_lib.verify_saved_models()   # 이번에 다룬 실험의 최종 가중치가 하나라도 없으면 여기서 멈추고 반납하지 않음
print("가중치 위치:", MODELS)
